In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# Supervised Learning

This module dives deeper into supervised learning concepts, including regression and classification techniques, model evaluation, and best practices.

## Learning Objectives

- Understand regression vs classification
- Implement linear and polynomial regression
- Use cross-validation for robust evaluation
- Perform hyperparameter tuning
- Apply supervised learning to real datasets

**Key Concepts:**
- **Regression**: Predicting continuous values (prices, temperatures, scores)
- **Classification**: Predicting categories (spam/not spam, species, labels)
- **Cross-Validation**: Testing on multiple train/test splits for reliable performance
- **Hyperparameter Tuning**: Finding best algorithm settings
- **Model Evaluation**: Measuring how well models perform


## Regression: California Housing Prices

**What is Regression?**
Regression predicts continuous numerical values (like house prices, temperatures, or test scores). Unlike classification (which predicts categories), regression predicts numbers.

**The California Housing Dataset:**
- **Features**: House characteristics (location, age, rooms, etc.)
- **Target**: Median house value (continuous number)
- **Task**: Predict house prices from features

**Why This Dataset?**
- Real-world application (real estate pricing)
- Multiple features (teaches feature engineering)
- Continuous target (regression problem)


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Scikit-learn: Machine learning library
from sklearn.datasets import fetch_california_housing, load_diabetes, make_regression  # Dataset loaders
from sklearn.linear_model import LinearRegression  # Linear regression algorithm
from sklearn.model_selection import cross_val_score  # Cross-validation

# Our custom utility functions
from src.models.supervised import split_data, evaluate_regressor, cross_validate_model  # Supervised learning utilities
from src.processing.preprocessing import scale_features  # Normalize features
import pandas as pd  # Data manipulation
import warnings  # Suppress warnings if needed

# ============================================
# LOADING THE DATASET: Try California Housing, Fallback to Diabetes
# ============================================

# Try to load California Housing dataset (requires internet download)
# If download fails (e.g., network issues), fall back to Diabetes dataset (built-in)
try:
    # fetch_california_housing() downloads California housing data from internet
    housing = fetch_california_housing()
    X = pd.DataFrame(housing.data, columns=housing.feature_names)  # Features: house characteristics
    y = pd.Series(housing.target)  # Target: median house value (continuous)
    dataset_name = "California Housing"
    target_name = housing.target_names[0] if hasattr(housing, 'target_names') else "Median House Value"
except Exception as e:
    # If download fails, use Diabetes dataset instead (no download needed)
    print(f"Could not download California Housing dataset: {e}")
    print("Using Diabetes dataset instead (no download required)...")
    diabetes = load_diabetes()  # Load built-in Diabetes dataset
    X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)  # Features: medical measurements
    y = pd.Series(diabetes.target)  # Target: disease progression (continuous)
    dataset_name = "Diabetes"
    target_name = "Disease Progression"

# ============================================
# EXPLORING THE DATASET: Understanding Our Data
# ============================================

print(f"\n{dataset_name} Dataset:")
print(X.head())  # Display first 5 rows
print(f"\nTarget: {target_name}")  # What we're predicting
print(f"Number of samples: {len(X)}")  # Total number of data points
print(f"Number of features: {X.shape[1]}")  # Number of input features

# ============================================
# FEATURE SCALING: Normalizing Features
# ============================================

# Scale features for better model performance
# Linear regression works better when features are on similar scales
X_scaled, scaler = scale_features(X, fit=True)  # Normalize: mean=0, std=1

# ============================================
# TRAIN/TEST SPLIT: Separating Data
# ============================================

# Split data into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.2, random_state=42)

# ============================================
# MODEL TRAINING: Linear Regression
# ============================================

# Create and train linear regression model
model = LinearRegression()  # Create model
model.fit(X_train, y_train)  # Train on training data

# ============================================
# MODEL EVALUATION: Measuring Performance
# ============================================

# Evaluate on test data (unseen during training)
results = evaluate_regressor(model, X_test, y_test)
# Returns dictionary with RMSE, MSE, MAE, etc.

# RMSE (Root Mean Squared Error): Average prediction error in same units as target
# Lower is better (0 = perfect predictions)
print(f"\nTest RMSE: {results['rmse']:.3f}")  # Test set error

# ============================================
# CROSS-VALIDATION: Robust Performance Estimate
# ============================================

# Cross-validation tests on multiple train/test splits
# More reliable than single split
cv_results = cross_validate_model(
    model,  # The model to evaluate
    X_scaled,  # All features (will be split internally)
    y,  # All targets (will be split internally)
    cv=5,  # 5-fold cross-validation (5 train/test splits)
    scoring='neg_mean_squared_error'  # Use negative MSE (scikit-learn convention)
)
# Returns dictionary with mean_score and std_score

# Convert negative MSE to RMSE
# -cv_results['mean_score'] converts back to positive MSE
# **0.5 takes square root to get RMSE
cv_rmse_mean = (-cv_results['mean_score'])**0.5  # Average RMSE across folds
cv_rmse_std = cv_results['std_score']**0.5  # Standard deviation of RMSE

print(f"CV RMSE: {cv_rmse_mean:.3f} (+/- {cv_rmse_std:.3f})")  # Average ± variability


Could not download California Housing dataset: HTTP Error 403: Forbidden
Using Diabetes dataset instead (no download required)...

Diabetes Dataset:
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019907 -0.017646  
1 -0.039493 -0.068332 -0.092204  
2 -0.002592  0.002861 -0.025930  
3  0.034309  0.022688 -0.009362  
4 -0.002592 -0.031988 -0.046641  

Target: Disease Progression
Number of samples: 442
Number of features: 10

Test RMSE: 53.853
CV RMSE: 54.709 (+/- 12.279)


## Classification: Breast Cancer Detection


In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from src.models.supervised import evaluate_classifier

# Load Breast Cancer dataset
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target)

print("Breast Cancer Dataset:")
print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Target: {cancer.target_names}")

# Split data
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

# Hyperparameter tuning with GridSearchCV
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, 10]
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")

# Evaluate best model
best_model = grid_search.best_estimator_
results = evaluate_classifier(best_model, X_test, y_test)
print(f"Accuracy: {results['accuracy']:.3f}")


Breast Cancer Dataset:
Features: 30
Samples: 569
Target: ['malignant' 'benign']

Best parameters: {'max_depth': 5, 'n_estimators': 100}
Accuracy: 0.965


## Summary

In this module, you learned:
- How to implement regression models
- How to perform classification tasks
- Cross-validation for robust evaluation
- Hyperparameter tuning with GridSearchCV
- Best practices for supervised learning

You're now ready to explore specific classifier algorithms in detail!
